# Lab 01: Exploring Bedrock Foundation Models

## Overview
In this lab you will invoke multiple foundation models on Amazon Bedrock using both the InvokeModel and Converse APIs. You will compare outputs across providers, tune inference parameters, and understand the difference between on-demand and provisioned throughput.

## Learning Objectives
- Invoke foundation models using `InvokeModel` and `Converse` APIs via boto3
- Compare outputs from Claude, Titan, Llama, and Mistral on the same prompt
- Tune inference parameters: temperature, top_p, top_k, max_tokens, stop sequences
- Stream model responses using `ConverseStream`
- Understand on-demand vs provisioned throughput pricing and use cases

## Exam Domain
**Selection & Implementation of Foundation Models (26%)** — This lab covers Task 1.1 (select appropriate foundation models) and Task 1.2 (implement foundation models using AWS services).

## Architecture Diagram
![Lab 01 Architecture](../assets/diagrams/lab-01-bedrock-foundation-models.png)

In [ ]:
%pip install boto3 --quiet

In [ ]:
import boto3
import json
import os
import time

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock = session.client("bedrock")
bedrock_runtime = session.client("bedrock-runtime")

# Verify access
models = bedrock.list_foundation_models()
print(f"Bedrock access confirmed. Found {len(models['modelSummaries'])} models available.")

## A. Listing Available Models

Amazon Bedrock is a fully managed service that provides access to foundation models from multiple providers through a single API. The available providers include:

- **Anthropic** — Claude family (text generation, analysis, coding)
- **Amazon** — Titan family (text, embeddings, image generation)
- **Meta** — Llama family (open-weight text generation)
- **Mistral AI** — Mistral and Mixtral (efficient text generation)
- **Cohere** — Command family (text generation, embeddings)
- **AI21 Labs** — Jamba family (text generation)
- **Stability AI** — Stable Diffusion (image generation)

Each model has different strengths, context window sizes, pricing, and supported features. The first step in working with Bedrock is understanding what models are available to you. Let's list them and group by provider.

In [ ]:
response = bedrock.list_foundation_models()
providers = {}
for model in response["modelSummaries"]:
    provider = model["providerName"]
    providers.setdefault(provider, []).append(model["modelId"])

for provider, model_ids in sorted(providers.items()):
    print(f"\n{provider} ({len(model_ids)} models):")
    for mid in model_ids[:3]:
        print(f"  - {mid}")
    if len(model_ids) > 3:
        print(f"  ... and {len(model_ids) - 3} more")

## B. InvokeModel API (Model-Specific Formats)

The `InvokeModel` API is the original way to call foundation models on Bedrock. It sends a raw request body to the model and returns a raw response body. The catch is that **each model provider has its own request and response format**.

This means you need to know the exact JSON schema for each model you want to call. Let's see this in practice by invoking Claude and Titan with the same prompt — notice how the request body and response parsing differ significantly between the two.

### Claude via InvokeModel

Anthropic's Claude models use the Messages API format. The request requires an `anthropic_version` field, a `messages` array with role/content pairs, and `max_tokens`. The response nests the generated text inside `content[0].text`.

In [ ]:
prompt = "Explain what Amazon Bedrock is in two sentences."

response = bedrock_runtime.invoke_model(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    contentType="application/json",
    accept="application/json",
    body=json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 256,
        "messages": [{"role": "user", "content": prompt}]
    })
)
result = json.loads(response["body"].read())
print("Claude:", result["content"][0]["text"])

### Llama 3 via InvokeModel

Meta's Llama 3 models use a different format from Claude. The prompt goes in a `prompt` field (not a messages array), and configuration uses `max_gen_len` and `temperature`. The response text is found at `generation` instead of `content[0].text`.

In [ ]:
response = bedrock_runtime.invoke_model(
    modelId="meta.llama3-8b-instruct-v1:0",
    contentType="application/json",
    accept="application/json",
    body=json.dumps({
        "prompt": prompt,
        "max_gen_len": 256,
        "temperature": 0.7
    })
)
result = json.loads(response["body"].read())
print("Llama 3:", result["generation"])

### The Problem with InvokeModel

As you can see, the same simple task — "send a prompt and get text back" — required completely different code for each model. If you wanted to add Mistral, you would need to learn yet another request format. This makes it difficult to:

- **Switch models** — changing providers means rewriting your integration code
- **Compare models** — you cannot easily run the same prompt across multiple models
- **Maintain code** — each model's format may change across versions

This is exactly the problem that the **Converse API** was designed to solve.

## C. Converse API (Unified Interface)

The **Converse API** provides a single, consistent interface for invoking any text model on Bedrock. Regardless of the provider, you use the same message format, the same inference configuration keys, and the same response structure.

Key advantages of the Converse API:
- **One format for all models** — the same code works with Claude, Titan, Llama, Mistral, and others
- **Simplified migration** — switching models requires changing only the `modelId` parameter
- **Consistent response parsing** — output is always at `response["output"]["message"]["content"][0]["text"]`

Let's create a reusable helper function and use it to compare four different models on the same prompt.

In [ ]:
def invoke_converse(model_id, prompt, max_tokens=256, temperature=0.7):
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature}
    )
    return response["output"]["message"]["content"][0]["text"]

models_to_compare = [
    ("us.anthropic.claude-sonnet-4-5-20250929-v1:0", "Claude Sonnet 4.5"),
    ("meta.llama3-8b-instruct-v1:0", "Llama 3 8B"),
    ("mistral.mistral-7b-instruct-v0:2", "Mistral 7B"),
]

prompt = "What are the three most important considerations when choosing a foundation model for a production application?"

for model_id, name in models_to_compare:
    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print(f"{'='*60}")
    try:
        output = invoke_converse(model_id, prompt)
        print(output)
    except Exception as e:
        print(f"Error: {e}")

## D. Parameter Tuning

Foundation models accept several inference parameters that control the style and structure of their output. Understanding these is critical for both the exam and real-world applications:

- **Temperature** (0.0–1.0) — Controls the randomness of token selection. At 0 the model always picks the most probable token (deterministic). At 1 it samples more broadly (creative, varied output). Use low temperature for factual tasks, high for creative tasks.
- **top_p** (0.0–1.0) — Nucleus sampling. The model considers only the smallest set of tokens whose cumulative probability is at least `p`. Lower values make output more focused; higher values allow more diversity. Typically you tune either temperature or top_p, not both.
- **max_tokens** — The maximum number of tokens the model will generate. This acts as a hard ceiling on response length and directly impacts cost.
- **Stop sequences** — Strings that cause the model to stop generating when encountered.

Let's see the effect of temperature on the same creative prompt.

### Temperature Comparison

Running the same creative prompt at three different temperature settings demonstrates how this parameter affects output variety. Run this cell multiple times — at temperature 0.0 you should get the same result each time, while temperature 1.0 will produce different outputs on each run.

In [ ]:
prompt = "Write a creative tagline for a cloud computing company."
model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

configs = [
    {"temperature": 0.0, "label": "temperature=0.0 (deterministic)"},
    {"temperature": 0.5, "label": "temperature=0.5 (balanced)"},
    {"temperature": 1.0, "label": "temperature=1.0 (creative)"},
]

for config in configs:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 100, "temperature": config["temperature"]}
    )
    output = response["output"]["message"]["content"][0]["text"]
    print(f"\n{config['label']}:")
    print(f"  {output}")

### top_p Comparison

The `top_p` parameter (nucleus sampling) works differently from temperature. Instead of scaling probabilities, it restricts the pool of candidate tokens. A `top_p` of 0.1 means the model only considers tokens in the top 10% of probability mass — producing very focused output. A `top_p` of 0.9 allows tokens from the top 90%, enabling more variety.

In [ ]:
prompt = "List three innovative uses of generative AI in healthcare."
model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

top_p_values = [
    {"topP": 0.1, "label": "top_p=0.1 (very focused)"},
    {"topP": 0.5, "label": "top_p=0.5 (balanced)"},
    {"topP": 0.9, "label": "top_p=0.9 (diverse)"},
]

for config in top_p_values:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 200, "topP": config["topP"]}
    )
    output = response["output"]["message"]["content"][0]["text"]
    print(f"\n{config['label']}:")
    print(f"  {output[:300]}...")

### max_tokens Effect

The `max_tokens` parameter sets a hard limit on how many tokens the model can generate. Setting it too low will cause the response to be cut off mid-sentence. Setting it too high increases cost without necessarily improving quality. Finding the right value depends on your use case — a chatbot might use 512, while a document summarizer might need 2048.

In [ ]:
prompt = "Explain the differences between supervised, unsupervised, and reinforcement learning."
model_id = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

token_limits = [50, 150, 500]

for limit in token_limits:
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": limit, "temperature": 0.3}
    )
    output = response["output"]["message"]["content"][0]["text"]
    stop_reason = response["stopReason"]
    print(f"\nmax_tokens={limit} (stop reason: {stop_reason}):")
    print(f"  {output[:200]}{'...' if len(output) > 200 else ''}")
    print(f"  [Response length: {len(output)} characters]")

## E. Streaming Responses

In production applications, users expect real-time feedback. Without streaming, the user sees nothing until the entire response is generated — which can take several seconds for longer outputs. **Streaming** returns tokens incrementally as they are produced, dramatically reducing perceived latency.

The `ConverseStream` API is the streaming version of the Converse API. Instead of returning a complete response, it returns an event stream that you iterate over. Each event contains either:
- `contentBlockDelta` — a chunk of generated text
- `messageStop` — indicates the model has finished generating (includes the stop reason)
- `metadata` — token usage statistics

In [ ]:
response = bedrock_runtime.converse_stream(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{"role": "user", "content": [{"text": "Explain RAG in 3 sentences."}]}],
    inferenceConfig={"maxTokens": 256}
)

print("Streaming response:")
for event in response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)
    if "messageStop" in event:
        print(f"\n\nStop reason: {event['messageStop']['stopReason']}")
    if "metadata" in event:
        usage = event["metadata"]["usage"]
        print(f"Tokens — input: {usage['inputTokens']}, output: {usage['outputTokens']}")

## Key Takeaways

1. Bedrock provides access to multiple foundation model providers through a single service
2. InvokeModel uses model-specific request formats; Converse API provides a unified interface
3. Temperature controls randomness (0=deterministic, 1=creative); top_p controls nucleus sampling
4. Streaming responses enable real-time user experiences with lower perceived latency
5. Model selection depends on task requirements: capability, latency, cost, and context window size

## Key Concepts

| Concept | Definition |
|---------|-----------|
| `InvokeModel` | Bedrock API that sends a prompt to a model using provider-specific request format |
| `Converse` | Unified Bedrock API that works with all models using a consistent message format |
| `ConverseStream` | Streaming version of Converse that returns response tokens incrementally |
| Temperature | Controls randomness of output; 0 = deterministic, 1 = maximum creativity |
| `top_p` | Nucleus sampling parameter; limits token selection to the smallest set with cumulative probability >= p |
| Provisioned Throughput | Reserved model capacity for predictable performance; charged hourly regardless of usage |
| On-Demand | Pay-per-token pricing with no reserved capacity; subject to throttling under high load |

## Exam Preparation — Common Exam Question Patterns

**Q: Which API provides a consistent interface across all Bedrock models?**
A: The Converse API. It uses a unified message format that works with all supported models, unlike InvokeModel which requires model-specific request bodies.

**Q: When should you use provisioned throughput instead of on-demand?**
A: When you need guaranteed latency and throughput for production workloads. Provisioned throughput reserves dedicated capacity, eliminating throttling risks — but you pay hourly regardless of usage.

**Q: What happens when you set temperature to 0?**
A: The model produces deterministic, most-likely outputs. Running the same prompt multiple times will return the same (or nearly the same) response.

**Q: Which Bedrock API requires model-specific request formatting?**
A: InvokeModel. Each provider (Anthropic, Amazon, Meta, Mistral) has its own JSON schema for both the request body and response format.

**Q: How do you reduce perceived latency for end users?**
A: Use streaming (ConverseStream) to return tokens as they are generated. The user sees the response building in real time instead of waiting for the full generation to complete.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Bedrock API (Claude Sonnet 4.5) | ~5K input + 5K output tokens | ~$0.30 |
| Bedrock API (Llama 3 8B) | ~2K input + 2K output tokens | ~$0.01 |
| Bedrock API (Mistral 7B) | ~2K input + 2K output tokens | ~$0.01 |
| **Total** | | **~$0.35** |